In [176]:
import pandas as pd
import warnings

## 1. Schools, Classes & Staff by Province

In [177]:
pub_cols = [
    'Province', 'Schools', 'Disadv_Schools', 'Classes', 'Classes_Pagoda',
    'Enrollment_Total', 'Enrollment_Girl',
    'Repeaters_Total', 'Repeaters_Girl',
    'TeachStaff_Total', 'TeachStaff_Female',
    'NonTeachStaff_Total', 'NonTeachStaff_Female',
    'TotalStaff_Total', 'TotalStaff_Female'
]

def clean_pub_main(df):
    df = df.iloc[3:-3].copy()
    df.columns = pub_cols

    df = df[df['Province'].notna()]
    df = df.set_index('Province')
    return df.apply(pd.to_numeric, errors='coerce').astype('Int64')

frames = []
for year, fname in [
    (2018, 'organized_education_data/public_schools_2018-2019.xlsx'),
    (2019, 'organized_education_data/public_schools_2019-2020.xlsx'),
    (2020, 'organized_education_data/public_schools_2020-2021.xlsx'),
]:
    raw = pd.read_excel(fname, sheet_name='Schools, Classes & Staff by Pro', header=None)
    df  = clean_pub_main(raw)
    df['Year'] = year
    frames.append(df)

combined_pub_main = pd.concat(frames)
cols = list(combined_pub_main.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_main = combined_pub_main[cols]

print('Shape:', combined_pub_main.shape)
combined_pub_main.head()

Shape: (75, 15)


,Year,Schools,Disadv_Schools,Classes,Classes_Pagoda,Enrollment_Total,Enrollment_Girl,Repeaters_Total,Repeaters_Girl,TeachStaff_Total,TeachStaff_Female,NonTeachStaff_Total,NonTeachStaff_Female,TotalStaff_Total,TotalStaff_Female
Province,,,,,,,,,,,,,,,
Banteay Meanchey,2018,818,15,4598,11,152590,76099,4051,1371,4782,2440,888,210,5670,2650
Battambang,2018,1133,17,7494,17,253481,125579,12244,4362,6946,4165,1811,672,8757,4837
Kampong Cham,2018,804,27,5846,111,221707,110363,14544,5235,5835,3390,1695,659,7530,4049
Kampong Chhnang,2018,466,0,3220,7,118144,59641,5080,1676,3498,1682,700,188,4198,1870
Kampong Speu,2018,603,2,4162,11,170222,83971,5931,2157,4471,1957,742,129,5213,2086


## 2. Enrollment & Repeaters by Grade

In [178]:
sub_hdrs = ['Enrollment_Total', 'Enrollment_Girl', 'Repeaters_Total', 'Repeaters_Girl']

# 49-column layout: Province + Grade_1..12 × 4 sub-cols (NO pre-built totals)
enroll_cols_pub = ['Province']
for i in range(1, 13):
    for s in sub_hdrs:
        enroll_cols_pub.append(f'Grade_{i}_{s}')


def _find_data_start(df):
    """Return row index of first real province (skips title + header rows)."""
    return next(
        i for i, v in enumerate(df.iloc[:, 0])
        if isinstance(v, str) and v not in ('Province',) and not v.startswith('Table')
    )


def parse_enroll_block(df, g_start):
    """13-column split sheet: Province + 3 grades × 4 sub-cols."""
    start = _find_data_start(df)
    
    # 1. Skip headers (start) AND last 3 rows (:-3)
    data = df.iloc[start:-3].copy().reset_index(drop=True)
    
    # 2. Filter out NaN provinces
    data = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    
    block_cols = ['Province']
    for g in range(g_start, g_start + 3):
        for s in sub_hdrs:
            block_cols.append(f'Grade_{g}_{s}')
            
    data = data.iloc[:, :len(block_cols)]
    data.columns = block_cols
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce').astype('Int64')


def parse_enroll_single(df):
    """49-column sheet: Province + 12 grades × 4 sub-cols."""
    start = _find_data_start(df)
    
    # 1. Skip headers (start) AND last 3 rows (:-3) <--- THIS IS THE FIX
    data = df.iloc[start:-3].copy().reset_index(drop=True)
    
    data = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    
    data = data.iloc[:, :49] 
    data.columns = enroll_cols_pub
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce').astype('Int64')


def add_totals(df):
    """Compute and append Total_* columns (sum across all 12 grades)."""
    for s in sub_hdrs:
        df[f'Total_{s}'] = df[[f'Grade_{g}_{s}' for g in range(1, 13)]].sum(axis=1)
    return df


# ── 2018-19: four split sheets ───────────────────────────────────────────────
f18 = 'organized_education_data/public_schools_2018-2019.xlsx'
enroll_18 = add_totals(pd.concat([
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters by Grad', header=None), 1),
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters (Gr1-4)', header=None), 4),
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters (Gr5-8)', header=None), 7),
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters(Gr9-12)', header=None), 10),
], axis=1))
enroll_18['Year'] = 2018

# ── 2019-20 & 2020-21: single sheet ─────────────────────────────────────────
enroll_19 = add_totals(parse_enroll_single(
    pd.read_excel('organized_education_data/public_schools_2019-2020.xlsx',
                  sheet_name='Enrollment & Repeaters by Grade', header=None)))
enroll_19['Year'] = 2019

enroll_20 = add_totals(parse_enroll_single(
    pd.read_excel('organized_education_data/public_schools_2020-2021.xlsx',
                  sheet_name='Enrollment & Repeaters by Grade', header=None)))
enroll_20['Year'] = 2020

# ── Combine ──────────────────────────────────────────────────────────────────
combined_pub_enroll = pd.concat([enroll_18, enroll_19, enroll_20])
cols = list(combined_pub_enroll.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_enroll = combined_pub_enroll[cols]

print('Shape:', combined_pub_enroll.shape)
combined_pub_enroll

Shape: (75, 53)


,Year,Grade_1_Enrollment_Total,Grade_1_Enrollment_Girl,Grade_1_Repeaters_Total,Grade_1_Repeaters_Girl,Grade_2_Enrollment_Total,Grade_2_Enrollment_Girl,Grade_2_Repeaters_Total,Grade_2_Repeaters_Girl,Grade_3_Enrollment_Total,Grade_3_Enrollment_Girl,Grade_3_Repeaters_Total,Grade_3_Repeaters_Girl,Grade_4_Enrollment_Total,Grade_4_Enrollment_Girl,Grade_4_Repeaters_Total,Grade_4_Repeaters_Girl,Grade_5_Enrollment_Total,Grade_5_Enrollment_Girl,Grade_5_Repeaters_Total,Grade_5_Repeaters_Girl,Grade_6_Enrollment_Total,Grade_6_Enrollment_Girl,Grade_6_Repeaters_Total,Grade_6_Repeaters_Girl,Grade_7_Enrollment_Total,Grade_7_Enrollment_Girl,Grade_7_Repeaters_Total,Grade_7_Repeaters_Girl,Grade_8_Enrollment_Total,Grade_8_Enrollment_Girl,Grade_8_Repeaters_Total,Grade_8_Repeaters_Girl,Grade_9_Enrollment_Total,Grade_9_Enrollment_Girl,Grade_9_Repeaters_Total,Grade_9_Repeaters_Girl,Grade_10_Enrollment_Total,Grade_10_Enrollment_Girl,Grade_10_Repeaters_Total,Grade_10_Repeaters_Girl,Grade_11_Enrollment_Total,Grade_11_Enrollment_Girl,Grade_11_Repeaters_Total,Grade_11_Repeaters_Girl,Grade_12_Enrollment_Total,Grade_12_Enrollment_Girl,Grade_12_Repeaters_Total,Grade_12_Repeaters_Girl,Total_Enrollment_Total,Total_Enrollment_Girl,Total_Repeaters_Total,Total_Repeaters_Girl
Province,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,18465,8748,1074,412,17049,8062,817,256,16656,7974,576,211,16637,7984,400,131,14577,7185,229,80,13516,6902,164,46,10773,5671,236,55,8656,4605,135,31,7348,3969,195,57,4956,2818,30,8,4061,2188,24,10,3272,1781,171,74,135966,67887,4051,1371
Battambang,2018,34641,16139,4037,1512,31386,14626,2462,852,29023,13746,1708,607,27697,13171,1124,383,25906,12794,673,224,23060,11620,498,163,18079,9334,575,163,13659,7482,301,90,11634,6218,458,177,8003,4458,88,31,6338,3554,72,28,5290,3077,248,132,234716,116219,12244,4362
Kampong Cham,2018,27237,12570,3946,1498,25534,11826,2868,1055,24421,11297,2210,773,23122,11100,1715,579,21483,10654,1221,435,19640,9948,747,266,16675,8572,546,200,13596,7492,370,98,11868,6497,377,119,9011,5093,175,54,7188,4100,40,9,6228,3458,329,149,206003,102607,14544,5235
Kampong Chhnang,2018,13264,6211,1395,502,12580,5995,1002,307,12692,6023,838,283,12748,6269,566,187,11976,5980,318,99,11014,5643,213,59,9456,4920,226,57,8017,4285,124,31,6923,3854,159,45,5024,2800,39,11,4075,2292,21,7,3461,1917,179,88,111230,56189,5080,1676
Kampong Speu,2018,19320,9030,1679,647,18429,8671,1210,440,18766,8879,915,322,19278,9461,615,222,18015,8938,430,187,17066,8617,497,182,13442,6760,181,44,11290,5972,83,15,9302,4773,217,65,6336,3227,19,2,4967,2559,9,0,4125,2092,76,31,160336,78979,5931,2157
Kampong Thom,2018,20195,9510,2600,1052,17887,8373,1713,629,17260,8287,1307,475,16184,7946,805,270,15067,7666,556,193,13972,7157,352,125,12478,6801,428,150,9677,5261,133,41,8348,4592,254,91,6047,3347,37,9,4964,2801,30,9,3952,2189,87,43,146031,73930,8302,3087
Kampot,2018,14257,6505,1212,415,13208,6179,681,229,13185,6234,599,206,12943,6191,366,122,12796,6193,271,96,12146,6077,202,58,10412,5223,262,63,9516,4933,133,31,8312,4280,307,76,6241,3226,71,19,5176,2694,34,9,4546,2311,91,42,122738,60046,4229,1366
Kandal,2018,29530,13885,3777,1413,27462,12497,2796,963,27160,12455,2310,830,27162,12834,1774,639,25747,12471,1208,437,22627,11131,777,227,19165,9614,634,160,15755,8302,293,112,13575,7309,355,98,10959,6109,118,34,8999,4998,59,22,7454,4063,422,185,235595,115668,14523,5120
Kep,2018,1041,510,94,41,921,428,49,13,878,437,47,20,859,450,34,15,728,364,17,4,677,334,22,7,620,310,29,9,505,252,5,3,481,231,17,5,368,195,7,2,377,197,1,0,325,180,2,1,7780,3888,324,120


## 3. Teaching Staff by Education Level

In [179]:
staff_cols = [
    'Province',
    'T_Primary', 'T_LSec', 'T_USec', 'T_Graduate', 'T_PostGrad', 'T_PhD',
    'NT_Primary', 'NT_LSec', 'NT_USec', 'NT_Graduate', 'NT_PostGrad', 'NT_PhD',
    'NoPed_Primary', 'NoPed_LSec', 'NoPed_USec', 'NoPed_Graduate', 'NoPed_PostGrad', 'NoPed_PhD',
]

def clean_pub_staff(df):
    start = next(
        i for i, v in enumerate(df.iloc[:, 1])
        if pd.notna(pd.to_numeric(v, errors='coerce'))
    )
    data = df.iloc[start:-3].copy().reset_index(drop=True)
    data = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    data.columns = staff_cols
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce').astype('Int64')

staff_sheet_names = {
    2018: 'School Staff by Education Leve',   # truncated tab name in 2018 file
    2019: 'School Staff by Education Level',
    2020: 'School Staff by Education Level',
}
fnames = {
    2018: 'organized_education_data/public_schools_2018-2019.xlsx',
    2019: 'organized_education_data/public_schools_2019-2020.xlsx',
    2020: 'organized_education_data/public_schools_2020-2021.xlsx',
}

staff_frames = []
for year in [2018, 2019, 2020]:
    raw = pd.read_excel(fnames[year], sheet_name=staff_sheet_names[year], header=None)
    df  = clean_pub_staff(raw)
    df['Year'] = year
    staff_frames.append(df)

combined_pub_staff = pd.concat(staff_frames)
cols = list(combined_pub_staff.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_staff = combined_pub_staff[cols]

print('Shape:', combined_pub_staff.shape)
combined_pub_staff.head()

Shape: (75, 19)


,Year,T_Primary,T_LSec,T_USec,T_Graduate,T_PostGrad,T_PhD,NT_Primary,NT_LSec,NT_USec,NT_Graduate,NT_PostGrad,NT_PhD,NoPed_Primary,NoPed_LSec,NoPed_USec,NoPed_Graduate,NoPed_PostGrad,NoPed_PhD
Province,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,49,779,2697,1188,51,0,14,204,393,232,42,0,0,0,10,1,0,0
Battambang,2018,79,1132,3450,2146,145,5,41,352,735,599,78,3,2,13,22,10,0,0
Kampong Cham,2018,127,1713,3043,914,38,0,59,531,893,186,29,0,6,17,34,5,0,0
Kampong Chhnang,2018,19,595,1912,932,40,0,10,158,341,169,22,0,0,1,1,4,0,0
Kampong Speu,2018,53,921,2174,1276,47,0,10,187,286,224,33,2,0,4,1,2,0,0


## 4. Student Flow Rates — Dropout / Promotion / Repetition

In [180]:
def parse_flow_sheet(fname, sheet, g_start, g_end):
    df    = pd.read_excel(fname, sheet_name=sheet, header=None)
    start = next(
        i for i, v in enumerate(df.iloc[:, 0])
        if isinstance(v, str) and v not in ('Province',) and not v.startswith('Table')
    )
    data = df.iloc[start:-3].copy().reset_index(drop=True)
    data = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    n = g_end - g_start + 1
    col_names = ['Province']
    for g in range(g_start, g_end + 1):
        col_names += [f'Grade_{g}_Promotion', f'Grade_{g}_Repetition', f'Grade_{g}_Dropout']
    data = data.iloc[:, :1 + n * 3].copy()
    data.columns = col_names
    data = data.set_index('Province')
    
    return data.apply(pd.to_numeric, errors='coerce')


# 2018-19: three separate sheets
f18 = 'organized_education_data/public_schools_2018-2019.xlsx'
flow_18 = pd.concat([
    parse_flow_sheet(f18, 'Flow Rates Both Gr1-4 2017-18',  1,  4),
    parse_flow_sheet(f18, 'Flow Rates Both Gr5-8 2017-18',  5,  8),
    parse_flow_sheet(f18, 'Flow Rates Both Gr9-12 2017-18', 9, 12),
], axis=1)
flow_18['Year'] = 2018

# 2019-20: single sheet (all grades)
flow_19 = parse_flow_sheet(
    'organized_education_data/public_schools_2019-2020.xlsx',
    'Student Flow Rates (Both) 2018-', 1, 12)
flow_19['Year'] = 2019

# 2020-21: single sheet (all grades)
flow_20 = parse_flow_sheet(
    'organized_education_data/public_schools_2020-2021.xlsx',
    'Student Flow Rates (Both) 2020-', 1, 12)
flow_20['Year'] = 2020

combined_pub_flow = pd.concat([flow_18, flow_19, flow_20])
cols = list(combined_pub_flow.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_flow = combined_pub_flow[cols]

print('Shape:', combined_pub_flow.shape)
combined_pub_flow

Shape: (75, 37)


,Year,Grade_1_Promotion,Grade_1_Repetition,Grade_1_Dropout,Grade_2_Promotion,Grade_2_Repetition,Grade_2_Dropout,Grade_3_Promotion,Grade_3_Repetition,Grade_3_Dropout,Grade_4_Promotion,Grade_4_Repetition,Grade_4_Dropout,Grade_5_Promotion,Grade_5_Repetition,Grade_5_Dropout,Grade_6_Promotion,Grade_6_Repetition,Grade_6_Dropout,Grade_7_Promotion,Grade_7_Repetition,Grade_7_Dropout,Grade_8_Promotion,Grade_8_Repetition,Grade_8_Dropout,Grade_9_Promotion,Grade_9_Repetition,Grade_9_Dropout,Grade_10_Promotion,Grade_10_Repetition,Grade_10_Dropout,Grade_11_Promotion,Grade_11_Repetition,Grade_11_Dropout,Grade_12_Promotion,Grade_12_Repetition,Grade_12_Dropout
Province,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,90.7,6.0,3.3,91.6,4.7,3.7,94.7,3.4,2.0,91.0,2.5,6.4,95.0,1.6,3.4,92.3,1.3,6.4,80.4,2.2,17.3,84.4,1.6,14.0,78.7,2.8,18.4,85.6,0.6,13.8,89.7,0.7,9.6,54.9,5.5,39.7
Battambang,2018,83.8,11.7,4.5,88.5,8.0,3.6,88.6,5.7,5.7,90.8,4.0,5.2,91.0,2.7,6.3,89.1,2.3,8.6,76.8,3.3,19.9,81.3,2.2,16.5,78.7,4.3,17.0,82.4,1.2,16.4,87.5,1.2,11.3,66.1,4.6,29.3
Kampong Cham,2018,81.6,14.2,4.2,84.4,10.9,4.7,86.5,8.9,4.6,88.0,7.4,4.6,89.8,5.8,4.4,91.2,3.9,4.9,80.1,3.3,16.6,82.1,2.6,15.2,79.5,3.3,17.2,80.0,2.0,18.1,89.9,0.6,9.5,56.6,5.3,38.1
Kampong Chhnang,2018,88.0,10.6,1.4,90.3,7.6,2.0,91.5,6.3,2.2,92.5,4.5,3.0,94.5,2.8,2.7,94.0,2.0,4.0,83.5,2.4,14.1,83.3,1.5,15.1,82.0,2.3,15.7,81.9,0.8,17.3,89.2,0.6,10.2,60.2,5.2,34.6
Kampong Speu,2018,89.3,8.7,2.0,92.5,6.3,1.2,92.7,4.5,2.8,93.2,3.3,3.5,92.7,2.4,4.9,93.4,3.1,3.5,79.5,1.3,19.2,80.6,0.7,18.6,75.1,2.2,22.7,79.4,0.3,20.3,89.4,0.2,10.4,84.4,1.9,13.7
Kampong Thom,2018,82.2,13.2,4.6,86.9,9.3,3.7,86.7,7.4,5.9,89.0,4.9,6.1,89.9,3.7,6.4,92.2,2.6,5.3,78.8,3.5,17.6,85.6,1.4,13.0,83.9,3.3,12.9,86.6,0.6,12.8,93.3,0.7,6.0,73.8,2.2,23.9
Kampot,2018,89.1,8.6,2.2,92.7,5.0,2.3,91.3,4.3,4.4,94.2,2.8,3.0,92.2,2.1,5.7,92.5,1.7,5.7,80.4,2.2,17.3,81.5,1.4,17.2,79.4,3.5,17.1,81.0,1.1,17.8,90.3,0.7,9.0,77.4,2.1,20.5
Kandal,2018,85.8,13.1,1.0,88.3,9.9,1.7,88.7,8.1,3.3,90.6,6.5,2.9,90.6,5.0,4.4,92.8,3.6,3.5,81.2,3.3,15.4,84.4,1.9,13.7,81.9,2.6,15.5,86.0,1.1,12.9,92.8,0.8,6.4,66.9,5.7,27.4
Kep,2018,90.6,9.8,NaN,98.0,5.8,NaN,93.8,5.3,0.9,89.2,4.3,6.5,89.8,2.3,7.8,86.0,3.1,10.8,73.5,4.3,22.2,78.5,0.8,20.6,78.6,3.4,17.9,88.9,1.7,9.5,91.2,0.3,8.5,94.5,0.8,4.7


# 5. school condition

In [181]:
school_20_21 = pd.read_excel("organized_education_data/public_schools_2020-2021.xlsx",sheet_name='Indicators on Schools - by Prov')
school_name = ['province','pupils per school','teacher per school','staff per school','building per school','room per school','classrom per school','classes per school','2-shift(%)','school in pagoda(%)','no water(%)','no toilet']
school_20_21.columns = school_name
school_20_21 = school_20_21[3:]
school_20_21 = school_20_21[:25]
school_20_21=school_20_21.set_index("province")
school_20_21 = school_20_21.astype('float64')
school_20_21.insert(0, 'year', 2020)

In [182]:
school_19_20 = pd.read_excel("organized_education_data/public_schools_2019-2020.xlsx",sheet_name='Indicators on Schools - by Prov')
school_19_20.columns = school_name
school_19_20 = school_19_20[3:]
school_19_20 = school_19_20[:25]
school_19_20=school_19_20.set_index("province")
school_19_20 = school_19_20.astype('float64')
school_19_20.insert(0, 'year', 2019)

In [183]:
school_18_19 = pd.read_excel("organized_education_data/public_schools_2019-2020.xlsx",sheet_name='Indicators on Schools - by Prov')
school_18_19.columns = school_name
school_18_19 = school_18_19[3:]
school_18_19 = school_18_19[:25]
school_18_19 = school_18_19.set_index("province")
school_18_19 = school_18_19.astype('float64')
school_18_19.insert(0, 'year', 2018)


In [184]:
school_pub = pd.concat([school_18_19,school_19_20,school_20_21])

In [185]:
combined_pub_main.shape

(75, 15)

In [186]:
combined_pub_enroll.shape

(75, 53)

In [187]:
combined_pub_staff.shape

(75, 19)

In [188]:
combined_pub_flow.shape

(75, 37)

In [189]:
school_pub.shape

(75, 12)

In [190]:
# 1. Reset the index so Province becomes a normal column
# 2. Set a MultiIndex using both Province and Year
combined_pub_main = combined_pub_main.reset_index().set_index(['Province', 'Year'])
combined_pub_enroll = combined_pub_enroll.reset_index().set_index(['Province', 'Year'])
combined_pub_staff = combined_pub_staff.reset_index().set_index(['Province', 'Year'])
combined_pub_flow = combined_pub_flow.reset_index().set_index(['Province', 'Year'])
school_pub = school_pub.reset_index().set_index(['province', 'year'])

# Now try the concat again
public_school_2018_2020 = pd.concat([
    combined_pub_main, 
    combined_pub_enroll, 
    combined_pub_staff, 
    combined_pub_flow, 
    school_pub
], axis=1)

In [191]:
public_school_2018_2020

,,Schools,Disadv_Schools,Classes,Classes_Pagoda,Enrollment_Total,Enrollment_Girl,Repeaters_Total,Repeaters_Girl,TeachStaff_Total,TeachStaff_Female,NonTeachStaff_Total,NonTeachStaff_Female,TotalStaff_Total,TotalStaff_Female,Grade_1_Enrollment_Total,Grade_1_Enrollment_Girl,Grade_1_Repeaters_Total,Grade_1_Repeaters_Girl,Grade_2_Enrollment_Total,Grade_2_Enrollment_Girl,Grade_2_Repeaters_Total,Grade_2_Repeaters_Girl,Grade_3_Enrollment_Total,Grade_3_Enrollment_Girl,Grade_3_Repeaters_Total,Grade_3_Repeaters_Girl,Grade_4_Enrollment_Total,Grade_4_Enrollment_Girl,Grade_4_Repeaters_Total,Grade_4_Repeaters_Girl,Grade_5_Enrollment_Total,Grade_5_Enrollment_Girl,Grade_5_Repeaters_Total,Grade_5_Repeaters_Girl,Grade_6_Enrollment_Total,Grade_6_Enrollment_Girl,Grade_6_Repeaters_Total,Grade_6_Repeaters_Girl,Grade_7_Enrollment_Total,Grade_7_Enrollment_Girl,Grade_7_Repeaters_Total,Grade_7_Repeaters_Girl,Grade_8_Enrollment_Total,Grade_8_Enrollment_Girl,Grade_8_Repeaters_Total,Grade_8_Repeaters_Girl,Grade_9_Enrollment_Total,Grade_9_Enrollment_Girl,Grade_9_Repeaters_Total,Grade_9_Repeaters_Girl,Grade_10_Enrollment_Total,Grade_10_Enrollment_Girl,Grade_10_Repeaters_Total,Grade_10_Repeaters_Girl,Grade_11_Enrollment_Total,Grade_11_Enrollment_Girl,Grade_11_Repeaters_Total,Grade_11_Repeaters_Girl,Grade_12_Enrollment_Total,Grade_12_Enrollment_Girl,Grade_12_Repeaters_Total,Grade_12_Repeaters_Girl,Total_Enrollment_Total,Total_Enrollment_Girl,Total_Repeaters_Total,Total_Repeaters_Girl,T_Primary,T_LSec,T_USec,T_Graduate,T_PostGrad,T_PhD,NT_Primary,NT_LSec,NT_USec,NT_Graduate,NT_PostGrad,NT_PhD,NoPed_Primary,NoPed_LSec,NoPed_USec,NoPed_Graduate,NoPed_PostGrad,NoPed_PhD,Grade_1_Promotion,Grade_1_Repetition,Grade_1_Dropout,Grade_2_Promotion,Grade_2_Repetition,Grade_2_Dropout,Grade_3_Promotion,Grade_3_Repetition,Grade_3_Dropout,Grade_4_Promotion,Grade_4_Repetition,Grade_4_Dropout,Grade_5_Promotion,Grade_5_Repetition,Grade_5_Dropout,Grade_6_Promotion,Grade_6_Repetition,Grade_6_Dropout,Grade_7_Promotion,Grade_7_Repetition,Grade_7_Dropout,Grade_8_Promotion,Grade_8_Repetition,Grade_8_Dropout,Grade_9_Promotion,Grade_9_Repetition,Grade_9_Dropout,Grade_10_Promotion,Grade_10_Repetition,Grade_10_Dropout,Grade_11_Promotion,Grade_11_Repetition,Grade_11_Dropout,Grade_12_Promotion,Grade_12_Repetition,Grade_12_Dropout,pupils per school,teacher per school,staff per school,building per school,room per school,classrom per school,classes per school,2-shift(%),school in pagoda(%),no water(%),no toilet
Banteay Meanchey,2018,818,15,4598,11,152590,76099,4051,1371,4782,2440,888,210,5670,2650,18465,8748,1074,412,17049,8062,817,256,16656,7974,576,211,16637,7984,400,131,14577,7185,229,80,13516,6902,164,46,10773,5671,236,55,8656,4605,135,31,7348,3969,195,57,4956,2818,30,8,4061,2188,24,10,3272,1781,171,74,135966,67887,4051,1371,49,779,2697,1188,51,0,14,204,393,232,42,0,0,0,10,1,0,0,90.7,6.0,3.3,91.6,4.7,3.7,94.7,3.4,2.0,91.0,2.5,6.4,95.0,1.6,3.4,92.3,1.3,6.4,80.4,2.2,17.3,84.4,1.6,14.0,78.7,2.8,18.4,85.6,0.6,13.8,89.7,0.7,9.6,54.9,5.5,39.7,191.0,5.9,7.0,1.4,6.6,4.6,5.7,34.2,2.3,79.6,47.6
Battambang,2018,1133,17,7494,17,253481,125579,12244,4362,6946,4165,1811,672,8757,4837,34641,16139,4037,1512,31386,14626,2462,852,29023,13746,1708,607,27697,13171,1124,383,25906,12794,673,224,23060,11620,498,163,18079,9334,575,163,13659,7482,301,90,11634,6218,458,177,8003,4458,88,31,6338,3554,72,28,5290,3077,248,132,234716,116219,12244,4362,79,1132,3450,2146,145,5,41,352,735,599,78,3,2,13,22,10,0,0,83.8,11.7,4.5,88.5,8.0,3.6,88.6,5.7,5.7,90.8,4.0,5.2,91.0,2.7,6.3,89.1,2.3,8.6,76.8,3.3,19.9,81.3,2.2,16.5,78.7,4.3,17.0,82.4,1.2,16.4,87.5,1.2,11.3,66.1,4.6,29.3,222.6,6.0,7.4,1.6,6.9,4.7,6.7,50.7,5.5,74.6,31.2
Kampong Cham,2018,804,27,5846,111,221707,110363,14544,5235,5835,3390,1695,659,7530,4049,27237,12570,3946,1498,25534,11826,2868,1055,24421,11297,2210,773,23122,11100,1715,579,21483,10654,1221,435,19640,9948,747,266,16675,8572,546,200,13596,7492,370,98,11868,6497,377,119,9011,5093,175,54,7188,4100,40,9,6228,345

In [192]:
pd.set_option("display.max_column",500)
pd.set_option("display.max_row",500)

In [193]:
public_school_2018_2020.shape

(75, 131)

In [ ]:
#public_school_2018_2020.to_csv("public_cleaned.csv")